# 各个包 to 配置

In [1]:
"""
MedicalGMU + X-ray + Clinical Tabular
- 影像(ResNet18)主导分类，临床特征仅做门控调制(GMU)
- 支持多任务辅助监督(aux_label)帮助影像学习形态特征
- 必须按病人(Group)划分训练/验证
"""

import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18

from sklearn.model_selection import KFold

current_dir = Path.cwd() #ipynb相当于cmd环境
root_dir = current_dir.parent 
text_path = root_dir / 'dataset' / 'text' / 'tabular_data.xlsx'
image_root = root_dir / 'dataset'  / 'image2' 
resnet_weight = root_dir / 'weights' / 'resnet18.pth'
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# device = torch.device("cuda:0")  
# weights = torch.tensor([1.0, 2.6132]).to(device)
# ========== 配置 ==========
CONFIG = {
	'excel_path': text_path,          # 包含 patient_id + feature_cols + aux_cols
	 'pkl_path': './clinical.pkl', 
	'image_root': image_root,               # 病人影像根目录
	'save_dir': './textonly2',
	'img_size': 224,
	'batch_size': 32,                            # 医学影像显存占用大，建议8-16
	'epochs': 100,
	'lr_backbone': 1e-5,                       # 影像backbone小学习率微调
	'lr_head': 1e-3,                           # 临床塔/分类头大学习率
	'weight_decay': 1e-4,
	'dropout': 0.4,
	'patience': 30,
	'use_aux_task': True,                      # 是否用 aux_label 做多任务辅助
	'aux_weight': 0.3,                         # 辅助损失权重
	'pos_weight': 3,                         # III型(正类)加权
	'seed': 42,
	'device': torch.device("cuda:0") ,
	'resnet_weight':resnet_weight
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Multi-Modal DATASET

## 预处理

In [2]:
# ========== 1. 数据预处理（生成 clinical.pkl） ==========

def preprocess_and_save(excel_path, pkl_path):
	"""严格使用你上传的 ChiariDataset 特征工程逻辑"""
	df = pd.read_excel(excel_path, header=1, nrows=280)
	df.columns = df.columns.str.strip()


	df_sorted = df.sort_values(by=['编号', '关节形态'], ascending=[True, False])
	patient_level =df.groupby('编号')['关节形态'].max().reset_index()
	# grouped = df.groupby('编号')
	# for patient_id, group_data in grouped:
	# 	print(f"=== 编号: {patient_id} 的分组 ===")
	# 	print(group_data)
	# 	print("----------------")
	# clinical_data = df.drop_duplicates(subset=['编号'], keep='first').copy()
	clinical_data = df_sorted.drop_duplicates(subset=['编号'], keep='first').copy()
	clinical_data = clinical_data.drop(columns= ['关节形态'])
	df = pd.merge(clinical_data, patient_level, on='编号')
	df['关节形态'] = df['关节形态'].apply(lambda x: 1 if x == 3 else 0)
	print(df.head())

	print(len(df))
	print(df.columns)
	data = pd.DataFrame()
	data['patient_id'] = df['编号'].astype(str).str.strip()
	# Categorical features (for Embedding)
	data['has_syrinx'] = df['合并脊髓空洞'].map({'是': 1, '有': 1, '否': 0, '无': 0}).fillna(0).astype(int)
	scale2_map = {'无痛': 0, '轻度疼痛': 1, '中度疼痛': 2, '重度疼痛': 3}
	data['scale2_1'] = df['疼痛分级'].map(scale2_map).fillna(0).astype(int)
	data['scale2_2'] = df['疼痛分级.1'].map(scale2_map).fillna(0).astype(int)
	data['scale2_3'] = df['疼痛分级.2'].map(scale2_map).fillna(0).astype(int)
	data['age']= df['年龄'].fillna(0).astype(int)
	data['gender']=df['性别'].map({'男': 1, '女': 0}).fillna(0).astype(int)
	asa_map = {'I': 0, 'II': 1, 'III': 2, 'IV': 3}
	data['asa']=df['ASA分级'].map(asa_map).fillna(0).astype(int)
	# Continuous features (for StandardScaler)
	data['syrinx_num'] = pd.to_numeric(df['脊髓空洞节段数量'], errors='coerce').fillna(0)
	data['syrinx_wide'] = pd.to_numeric(df['脊髓空洞最大宽度（只计算有空洞的）'], errors='coerce').fillna(0)
	data['CTH'] = pd.to_numeric(df['小脑扁桃体疝出程度'], errors='coerce').fillna(0)
	
	cisterna = pd.to_numeric(df['枕大池容积'], errors='coerce')
	# cisterna_max = cisterna.max()
	data['cisterna_flipped'] = (-cisterna).fillna(0)
	
	data['scale1'] = pd.to_numeric(df['量表1总分'], errors='coerce').fillna(0)
	data['scale3'] = pd.to_numeric(df['术前量表3总分'], errors='coerce').fillna(100)  # ADL，缺失=满分
	joa = pd.to_numeric(df['术前量表4总分'], errors='coerce')
	data['joa_flipped'] = (17 - joa).fillna(0)

	# 保存（标签从影像文件夹解析，这里不存label）
	data.to_pickle(pkl_path)
	print(f"[Preprocess] Saved clinical features to {pkl_path}")
	return data


In [ ]:
data = preprocess_and_save(CONFIG['excel_path'], CONFIG['pkl_path'])

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
	accuracy_score, precision_score, recall_score, 
	f1_score, roc_auc_score, confusion_matrix, average_precision_score,
)
from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image


# ==================== 1. 模型 ====================

class ClinicalEncoder(nn.Module):
	"""
	临床表格编码器（与你原代码完全一致）
	"""
	def __init__(self, cont_dim=7, out_dim=16):
		super().__init__()
		self.embeddings = nn.ModuleDict({
			'has_syrinx': nn.Embedding(2, 2),      # 2 classes
			'scale2_1': nn.Embedding(4, 3),         # 4 classes (0-3)
			'scale2_2': nn.Embedding(4, 3),
			'scale2_3': nn.Embedding(4, 3),
			'gender': nn.Embedding(2,2),
			'asa': nn.Embedding(4,3),		
		})
		cat_dim = 2 + 3 + 3 + 3  +3+2# 16

		self.cont_proj = nn.Sequential(
			nn.Linear(cont_dim, 16),
			nn.BatchNorm1d(16),
			nn.ReLU(),
			nn.Dropout(0.2)
		)
		self.fusion = nn.Sequential(
			nn.Linear(cat_dim + 16, out_dim),
			nn.LayerNorm(16),
			nn.Dropout(p=0.1)
		)

	def forward(self, cat_features, cont_features):
		embeds = []
		for key, emb in self.embeddings.items():
			embeds.append(emb(cat_features[key]))
		cat_feat = torch.cat(embeds, dim=-1)  # [B, 11]
		cont_feat = self.cont_proj(cont_features)  # [B, 16]
		combined = torch.cat([cat_feat, cont_feat], dim=-1)  # [B, 27]
		return self.fusion(combined)  # [B, 16]


class TabNet(nn.Module):
	"""
	纯表格分类网络：ClinicalEncoder + 分类头
	"""
	def __init__(self, num_cont=8, clin_out=16, num_classes=2):
		super().__init__()
		self.clinical_encoder = ClinicalEncoder(cont_dim=num_cont, out_dim=clin_out)
		self.classifier = nn.Sequential(
			nn.Linear(clin_out, 32),
			nn.BatchNorm1d(32),
			nn.ReLU(),
			nn.Dropout(0.3),
			nn.Linear(32, num_classes)
		)
		self._init_weights()

	def _init_weights(self):
		for m in self.modules():
			if isinstance(m, nn.Linear):
				nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
				if m.bias is not None:
					nn.init.zeros_(m.bias)
			elif isinstance(m, nn.Embedding):
				nn.init.normal_(m.weight, std=0.1)

	def forward(self, cat_features, cont_features):
		"""
		Args:
			cat_features: dict of [B] LongTensor
			cont_features: [B, num_cont] FloatTensor
		Returns:
			logits: [B, num_classes]
		"""
		clin_feat = self.clinical_encoder(cat_features, cont_features)  # [B, 16]
		return self.classifier(clin_feat)  # [B, 2]


# ==================== 2. 数据集（无图像） ====================

class ChiariTabDataset(Dataset):
	def __init__(self, pkl_path, image_root, patient_ids, scaler=None, fit_scaler=False):
		"""
		纯表格数据集：以 image_root 的目录结构确定病人/关节/标签，
		但不再加载任何影像，每个关节只保留一条记录。
		"""
		self.root_dir = Path(image_root)
		self.samples = []

		# 加载临床表格
		df = pd.read_pickle(pkl_path)
		df['patient_id'] = df['patient_id'].astype(str).str.strip()
		df_indexed = df.set_index('patient_id')

		self.cat_cols = ['has_syrinx', 'scale2_1', 'scale2_2', 'scale2_3','gender','asa']
		self.cont_cols = ['syrinx_num', 'syrinx_wide', 'CTH',
						  'cisterna_flipped', 'scale1', 'scale3', 'joa_flipped','age']

		# Continuous 标准化
		if fit_scaler:
			self.scaler = StandardScaler()
			df[self.cont_cols] = self.scaler.fit_transform(df[self.cont_cols])
		else:
			self.scaler = scaler
			if scaler is not None:
				df[self.cont_cols] = scaler.transform(df[self.cont_cols])

		# 遍历目录结构：病人 -> 关节文件夹
		for p_id in patient_ids:
			p_id_str = str(p_id).strip()
			if p_id_str not in df_indexed.index:
				continue

			row = df_indexed.loc[p_id_str]
			cat_dict = {c: int(row[c]) for c in self.cat_cols}
			cont_arr = row[self.cont_cols].values.astype(float)

			patient_folder = self.root_dir / p_id_str
			if not patient_folder.is_dir():
				continue

			for side_folder in patient_folder.iterdir():
				if not side_folder.is_dir():
					continue
				folder_name = side_folder.name

				side = 'left' if '左' in folder_name else 'right'

				# 标签解析
				if '3型' in folder_name or 'III型' in folder_name:
					label = 1
				else:
					label = 0

				# 每个关节只保留一条记录（不再遍历影像文件）
				self.samples.append({
					'label': label,
					'patient_id': p_id_str,
					'side': side,
					'cat_dict': cat_dict,
					'cont_arr': cont_arr.copy()
				})

		print(f"[Dataset] 病人: {len(set(s['patient_id'] for s in self.samples))}, "
			  f"关节样本: {len(self.samples)}, "
			  f"正类(III型): {sum(s['label'] for s in self.samples)}")

	def __len__(self):
		return len(self.samples)

	def __getitem__(self, idx):
		s = self.samples[idx]
		label = torch.tensor(s['label'], dtype=torch.long)

		cat_features = {
			k: torch.tensor(v, dtype=torch.long)
			for k, v in s['cat_dict'].items()
		}
		cont_features = torch.tensor(s['cont_arr'], dtype=torch.float32)
		joint_id = f"{s['patient_id']}_{s['side']}"

		# 返回 5 个元素：label, cat, cont, patient_id, joint_id（无图像）
		return label, cat_features, cont_features, s['patient_id'], joint_id


def collate_fn(batch):
	"""纯表格 collate"""
	labels = torch.stack([b[0] for b in batch])
	cat_keys = batch[0][1].keys()
	cat_batch = {k: torch.stack([b[1][k] for b in batch]) for k in cat_keys}
	cont_batch = torch.stack([b[2] for b in batch])
	patient_ids = [b[3] for b in batch]
	joint_ids = [b[4] for b in batch]

	return labels, cat_batch, cont_batch, patient_ids, joint_ids


# ==================== 3. 评估（无图像） ====================

@torch.no_grad()
def evaluate_joint_level(model, dataloader, device, search_threshold=False, best_t=None):
	"""
	关节级评估：纯表格版（每个关节只有一条记录，退化为样本级评估）
	"""
	model.eval()
	joint_probs = defaultdict(list)
	joint_labels = {}

	for labels, cat_batch, cont_batch, patient_ids, joint_ids in dataloader:
		cat_batch = {k: v.to(device) for k, v in cat_batch.items()}
		cont_batch = cont_batch.to(device)

		logits = model(cat_batch, cont_batch)

		if logits.dim() == 1 or logits.size(-1) == 1:
			probs = torch.sigmoid(logits).cpu().numpy()
			if probs.ndim == 2:
				probs = probs.squeeze(-1)
		else:
			probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

		for i in range(len(joint_ids)):
			jid = joint_ids[i]
			joint_probs[jid].append(float(probs[i]))
			joint_labels[jid] = int(labels[i].item())

	# 聚合（每个关节平均，虽然纯表格版通常只有一条）
	final_labels, final_probs = [], []
	for jid in sorted(joint_probs.keys()):
		avg_prob = np.mean(joint_probs[jid])
		final_probs.append(avg_prob)
		final_labels.append(joint_labels[jid])

	final_labels = np.array(final_labels)
	final_probs = np.array(final_probs)

	if search_threshold:
		from sklearn.metrics import roc_curve
		fpr, tpr, thresholds = roc_curve(final_labels, final_probs)
		tnr = 1 - fpr
		youden_j = tpr + tnr - 1
		best_idx = np.argmax(youden_j)
		threshold = thresholds[best_idx]
	else:
		threshold = best_t if best_t is not None else 0.5

	final_preds = (final_probs >= threshold).astype(int)

	# 指标
	acc = accuracy_score(final_labels, final_preds)
	prec = precision_score(final_labels, final_preds, zero_division=0)
	rec = recall_score(final_labels, final_preds, zero_division=0)
	f1 = f1_score(final_labels, final_preds, zero_division=0)

	try:
		auc = roc_auc_score(final_labels, final_probs)
	except ValueError:
		auc = 0.5
	try:
		ap = average_precision_score(final_labels, final_probs)
	except ValueError:
		ap = 0.0

	try:
		tn, fp, fn, tp = confusion_matrix(final_labels, final_preds).ravel()
		sens = tp / (tp + fn) if (tp + fn) > 0 else 0
		spec = tn / (tn + fp) if (tn + fp) > 0 else 0
		ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
		npv = tn / (tn + fn) if (tn + fn) > 0 else 0
	except ValueError:
		tn = fp = fn = tp = 0
		sens = spec = ppv = npv = 0

	metrics = {
		'auc': float(auc),
		'ap': float(ap),
		'threshold': float(threshold),
		'accuracy': float(acc),
		'precision': float(prec),
		'recall': float(rec),
		'f1': float(f1),
		'sensitivity': float(sens),
		'specificity': float(spec),
		'ppv': float(ppv),
		'npv': float(npv),
		'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
		'num_joints': len(final_labels),
		'num_positive': int(final_labels.sum()),
	}

	return metrics, final_labels.tolist(), final_preds.tolist(), final_probs.tolist()


# ==================== 4. 训练主函数 ====================

def main_train(CONFIG):
	print("=" * 60)
	print("TabNet (Pure Clinical Table) Training")
	print("=" * 60)

	# 1. 获取所有病人ID（仍以影像文件夹为准，确定结构）
	all_pids = [p.name for p in Path(CONFIG['image_root']).iterdir() if p.is_dir()]
	all_pids = sorted(list(set(all_pids)))
	print(f"总病人: {len(all_pids)}")

	kf = KFold(n_splits=5, shuffle=True, random_state=42)
	fold_results = []

	for fold, (train_idx, val_idx) in enumerate(kf.split(all_pids)):
		print(f"\n{'='*40}")
		print(f"开始训练第 {fold+1} 折...")
		print(f"{'='*40}")

		train_pids = [all_pids[i] for i in train_idx]
		val_pids = [all_pids[i] for i in val_idx]
		print(f"Train: {len(train_pids)}, Val: {len(val_pids)}")

		# 2. Dataset & DataLoader（无图像，无 transform）
		train_ds = ChiariTabDataset(
			CONFIG['pkl_path'], CONFIG['image_root'], train_pids,
			fit_scaler=True
		)
		val_ds = ChiariTabDataset(
			CONFIG['pkl_path'], CONFIG['image_root'], val_pids,
			scaler=train_ds.scaler, fit_scaler=False
		)

		train_loader = DataLoader(
			train_ds, batch_size=CONFIG['batch_size'],
			shuffle=True, num_workers=4,
			collate_fn=collate_fn, pin_memory=True, drop_last=True
		)
		val_loader = DataLoader(
			val_ds, batch_size=CONFIG['batch_size'],
			shuffle=False, num_workers=4,
			collate_fn=collate_fn, pin_memory=True
		)

		# 3. 模型（纯表格）
		device = CONFIG['device']
		model = TabNet(num_cont=8, clin_out=16, num_classes=2).to(device)

		# 优化器（不再分层，所有参数一起训练）
		optimizer = torch.optim.AdamW(
			model.parameters(),
			lr=CONFIG['lr_head'],
			weight_decay=CONFIG['weight_decay']
		)
		scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
			optimizer, mode='max', factor=0.5, patience=10, verbose=True
		)

		# 损失（CE 多分类）
		class_weights = torch.tensor([1.0, CONFIG['pos_weight']]).to(device)
		criterion = nn.CrossEntropyLoss(weight=class_weights)

		# 4. 训练循环
		best_auc, best_epoch, no_improve = 0, 0, 0

		for epoch in range(CONFIG['epochs']):
			model.train()
			epoch_loss = 0

			for labels, cat_batch, cont_batch, _, _ in train_loader:
				labels = labels.to(device)
				cat_batch = {k: v.to(device) for k, v in cat_batch.items()}
				cont_batch = cont_batch.to(device)

				optimizer.zero_grad()
				logits = model(cat_batch, cont_batch)
				loss = criterion(logits, labels)
				loss.backward()
				torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
				optimizer.step()
				epoch_loss += loss.item()

			# 验证
			val_metrics, _, _, _ = evaluate_joint_level(
				model, val_loader, device, search_threshold=False, best_t=0.25
			)
			val_auc = val_metrics['auc']
			val_acc = val_metrics['accuracy']
			scheduler.step(val_auc)

			print(f"Epoch {epoch:03d} | Loss: {epoch_loss/len(train_loader):.4f} | "
				  f"Val AUC: {val_auc:.3f} | Acc: {val_acc:.3f} | "
				  f"Sen: {val_metrics['sensitivity']:.3f} | "
				  f"Spe: {val_metrics['specificity']:.3f} | "
				  f"PR: {val_metrics['precision']:.3f} | "
				  f"REC: {val_metrics['recall']:.3f} | "
				  f"F1: {val_metrics['f1']:.3f}")

			# 保存最优
			if val_auc > best_auc:
				best_auc, best_epoch = val_auc, epoch
				no_improve = 0
				save_dict = {
					'model_state': model.state_dict(),
					'scaler': train_ds.scaler,
					'cat_cols': train_ds.cat_cols,
					'cont_cols': train_ds.cont_cols,
					'threshold': val_metrics['threshold'],
					'metrics': val_metrics,
					'config': CONFIG,
					'train_patient_ids': train_pids,
					'val_patient_ids': val_pids,
				}
				os.makedirs(CONFIG['save_dir'], exist_ok=True)
				torch.save(save_dict, os.path.join(CONFIG['save_dir'], f'{fold+1}_tabnet_best.pt'))
				print(f"  >>> Saved best (AUC={best_auc:.3f})")
			else:
				no_improve += 1

			if no_improve >= CONFIG['patience']:
				print(f"\n早停于 Epoch {epoch} (Best: Epoch {best_epoch}, AUC={best_auc:.3f})")
				break

		fold_results.append(val_metrics)

	# 5. 汇总
	print(f"\n{'='*60}")
	print("5-Fold Results Summary")
	print(f"{'='*60}")
	keys = ['accuracy', 'auc', 'ap', 'f1', 'precision', 'recall', 'sensitivity', 'specificity']
	for k in keys:
		vals = [m[k] for m in fold_results]
		print(f"{k:<12}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")

	print(f"\n训练完成！")




In [ ]:
if __name__ == '__main__':
	main_train(CONFIG)

TabNet (Pure Clinical Table) Training
总病人: 215

开始训练第 1 折...
Train: 172, Val: 43
[Dataset] 病人: 172, 关节样本: 344, 正类(III型): 104
[Dataset] 病人: 43, 关节样本: 86, 正类(III型): 31


/media/sdc1/25yhlu/miniconda3/envs/AOJ/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 000 | Loss: 0.8769 | Val AUC: 0.515 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
  >>> Saved best (AUC=0.515)
Epoch 001 | Loss: 0.8254 | Val AUC: 0.437 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 002 | Loss: 0.8202 | Val AUC: 0.431 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 003 | Loss: 0.8133 | Val AUC: 0.426 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 004 | Loss: 0.8171 | Val AUC: 0.426 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 005 | Loss: 0.7394 | Val AUC: 0.433 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 006 | Loss: 0.8317 | Val AUC: 0.390 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 007 | Loss: 0.7449 | Val AUC: 0.367 | Acc: 0.360 | Sen: 1.000 | Spe: 0.000 | PR: 0.360 | REC: 1.000 | F1: 0.530
Epoch 008 | Loss: 0.7979 | 

/media/sdc1/25yhlu/miniconda3/envs/AOJ/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 000 | Loss: 0.9461 | Val AUC: 0.641 | Acc: 0.628 | Sen: 0.364 | Spe: 0.719 | PR: 0.308 | REC: 0.364 | F1: 0.333
  >>> Saved best (AUC=0.641)
Epoch 001 | Loss: 0.8788 | Val AUC: 0.750 | Acc: 0.488 | Sen: 0.909 | Spe: 0.344 | PR: 0.323 | REC: 0.909 | F1: 0.476
  >>> Saved best (AUC=0.750)
Epoch 002 | Loss: 0.8678 | Val AUC: 0.727 | Acc: 0.349 | Sen: 0.909 | Spe: 0.156 | PR: 0.270 | REC: 0.909 | F1: 0.417
Epoch 003 | Loss: 0.7850 | Val AUC: 0.680 | Acc: 0.326 | Sen: 0.909 | Spe: 0.125 | PR: 0.263 | REC: 0.909 | F1: 0.408
Epoch 004 | Loss: 0.8388 | Val AUC: 0.682 | Acc: 0.279 | Sen: 0.909 | Spe: 0.062 | PR: 0.250 | REC: 0.909 | F1: 0.392
Epoch 005 | Loss: 0.8379 | Val AUC: 0.656 | Acc: 0.279 | Sen: 0.909 | Spe: 0.062 | PR: 0.250 | REC: 0.909 | F1: 0.392
Epoch 006 | Loss: 0.8336 | Val AUC: 0.636 | Acc: 0.279 | Sen: 0.909 | Spe: 0.062 | PR: 0.250 | REC: 0.909 | F1: 0.392
Epoch 007 | Loss: 0.6997 | Val AUC: 0.612 | Acc: 0.279 | Sen: 0.909 | Spe: 0.062 | PR: 0.250 | REC: 0.909 | F1: 0.39

/media/sdc1/25yhlu/miniconda3/envs/AOJ/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 000 | Loss: 0.9176 | Val AUC: 0.548 | Acc: 0.372 | Sen: 1.000 | Spe: 0.036 | PR: 0.357 | REC: 1.000 | F1: 0.526
  >>> Saved best (AUC=0.548)
Epoch 001 | Loss: 0.8273 | Val AUC: 0.546 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.517
Epoch 002 | Loss: 0.8390 | Val AUC: 0.574 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.517
  >>> Saved best (AUC=0.574)
Epoch 003 | Loss: 0.8380 | Val AUC: 0.570 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.517
Epoch 004 | Loss: 0.8513 | Val AUC: 0.552 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.517
Epoch 005 | Loss: 0.7993 | Val AUC: 0.519 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.517
Epoch 006 | Loss: 0.7816 | Val AUC: 0.502 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.517
Epoch 007 | Loss: 0.7841 | Val AUC: 0.480 | Acc: 0.349 | Sen: 1.000 | Spe: 0.000 | PR: 0.349 | REC: 1.000 | F1: 0.51

/media/sdc1/25yhlu/miniconda3/envs/AOJ/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 000 | Loss: 0.8895 | Val AUC: 0.503 | Acc: 0.291 | Sen: 1.000 | Spe: 0.032 | PR: 0.274 | REC: 1.000 | F1: 0.430
  >>> Saved best (AUC=0.503)
Epoch 001 | Loss: 0.8638 | Val AUC: 0.588 | Acc: 0.314 | Sen: 1.000 | Spe: 0.063 | PR: 0.280 | REC: 1.000 | F1: 0.438
  >>> Saved best (AUC=0.588)
Epoch 002 | Loss: 0.7704 | Val AUC: 0.576 | Acc: 0.314 | Sen: 1.000 | Spe: 0.063 | PR: 0.280 | REC: 1.000 | F1: 0.438
Epoch 003 | Loss: 0.7929 | Val AUC: 0.586 | Acc: 0.267 | Sen: 1.000 | Spe: 0.000 | PR: 0.267 | REC: 1.000 | F1: 0.422
Epoch 004 | Loss: 0.7889 | Val AUC: 0.608 | Acc: 0.267 | Sen: 1.000 | Spe: 0.000 | PR: 0.267 | REC: 1.000 | F1: 0.422
  >>> Saved best (AUC=0.608)
Epoch 005 | Loss: 0.7219 | Val AUC: 0.599 | Acc: 0.267 | Sen: 1.000 | Spe: 0.000 | PR: 0.267 | REC: 1.000 | F1: 0.422
Epoch 006 | Loss: 0.7750 | Val AUC: 0.590 | Acc: 0.267 | Sen: 1.000 | Spe: 0.000 | PR: 0.267 | REC: 1.000 | F1: 0.422
Epoch 007 | Loss: 0.7637 | Val AUC: 0.595 | Acc: 0.267 | Sen: 1.000 | Spe: 0.000 | PR: 

/media/sdc1/25yhlu/miniconda3/envs/AOJ/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 000 | Loss: 0.9971 | Val AUC: 0.558 | Acc: 0.337 | Sen: 1.000 | Spe: 0.000 | PR: 0.337 | REC: 1.000 | F1: 0.504
  >>> Saved best (AUC=0.558)
Epoch 001 | Loss: 0.9461 | Val AUC: 0.552 | Acc: 0.360 | Sen: 1.000 | Spe: 0.035 | PR: 0.345 | REC: 1.000 | F1: 0.513
Epoch 002 | Loss: 0.8915 | Val AUC: 0.608 | Acc: 0.384 | Sen: 1.000 | Spe: 0.070 | PR: 0.354 | REC: 1.000 | F1: 0.523
  >>> Saved best (AUC=0.608)
Epoch 003 | Loss: 0.8236 | Val AUC: 0.579 | Acc: 0.407 | Sen: 1.000 | Spe: 0.105 | PR: 0.362 | REC: 1.000 | F1: 0.532
Epoch 004 | Loss: 0.8756 | Val AUC: 0.562 | Acc: 0.407 | Sen: 1.000 | Spe: 0.105 | PR: 0.362 | REC: 1.000 | F1: 0.532
Epoch 005 | Loss: 0.8831 | Val AUC: 0.553 | Acc: 0.384 | Sen: 1.000 | Spe: 0.070 | PR: 0.354 | REC: 1.000 | F1: 0.523
Epoch 006 | Loss: 0.7874 | Val AUC: 0.542 | Acc: 0.360 | Sen: 1.000 | Spe: 0.035 | PR: 0.345 | REC: 1.000 | F1: 0.513
Epoch 007 | Loss: 0.8081 | Val AUC: 0.541 | Acc: 0.360 | Sen: 1.000 | Spe: 0.035 | PR: 0.345 | REC: 1.000 | F1: 0.51

# EVA

In [ ]:
import os
import pickle
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ==================== 5-Fold 权重加载与约登阈值推理 ====================

@torch.no_grad()
def inference_5fold_with_youden(CONFIG, save_pkl_path=None):
    print("=" * 65)
    print("TabNet 5-Fold Inference (Youden's Index Optimal Threshold)")
    print("=" * 65)

    device = CONFIG['device']
    fold_metrics_list = []
    oof_fold_dict = {}  # 用于保存与多模型对比脚本兼容的 5fold_results.pkl

    # 如果 checkpoint 中未保存 val_patient_ids，则使用相同的 KFold 种子兜底恢复
    all_pids = sorted(list(set([p.name for p in Path(CONFIG['image_root']).iterdir() if p.is_dir()])))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    kf_splits = list(kf.split(all_pids))

    for fold in range(5):
        ckpt_path = os.path.join(CONFIG['save_dir'], f'{fold+1}_tabnet_best.pt')
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(f"未找到第 {fold+1} 折权重文件: {ckpt_path}")

        # 1. 加载 Checkpoint（包含模型权重、训练好的 scaler 和验证集病人 ID）
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        val_pids = ckpt.get('val_patient_ids', [all_pids[i] for i in kf_splits[fold][1]])
        saved_scaler = ckpt['scaler']

        # 2. 构建验证集 DataLoader（直接复用保存的 scaler，无需重新构建 train_ds）
        val_ds = ChiariTabDataset(
            CONFIG['pkl_path'],
            CONFIG['image_root'],
            val_pids,
            scaler=saved_scaler,
            fit_scaler=False
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=CONFIG['batch_size'],
            shuffle=False,
            num_workers=4,
            collate_fn=collate_fn,
            pin_memory=True
        )

        # 3. 实例化模型并加载最优权重
        model = TabNet(num_cont=8, clin_out=16, num_classes=2).to(device)
        model.load_state_dict(ckpt['model_state'])
        model.eval()

        # 4. 调用你现有的 evaluate_joint_level，开启 search_threshold=True (约登指数)
        val_metrics, labels_list, preds_list, probs_list = evaluate_joint_level(
            model, val_loader, device, search_threshold=False
        )
        fold_metrics_list.append(val_metrics)

        # 提取排好序的 joint_ids，方便后续跨模型配对统计检验 (DeLong / McNemar)
        joint_ids_set = set()
        for _, _, _, _, jids in val_loader:
            joint_ids_set.update(jids)
        sorted_jids = sorted(list(joint_ids_set))

        oof_fold_dict[fold] = (
            np.array(labels_list, dtype=int),
            np.array(probs_list, dtype=float),
            sorted_jids
        )

        print(
            f"Fold {fold+1} | Youden Thresh: {val_metrics['threshold']:.4f} | "
            f"AUC: {val_metrics['auc']:.3f} | AP: {val_metrics['ap']:.3f} | "
            f"F1: {val_metrics['f1']:.3f} | Acc: {val_metrics['accuracy']:.3f} | "
            f"Sen: {val_metrics['sensitivity']:.3f} | Spe: {val_metrics['specificity']:.3f}"
        )

    # 5. 汇总 5-Fold 均值 ± 标准差
    print(f"\n{'='*65}")
    print("5-Fold Results Summary (Youden's Index Threshold)")
    print(f"{'='*65}")
    keys = [
        'threshold', 'auc', 'ap', 'f1',
        'specificity', 'sensitivity', 'accuracy', 'precision', 'recall'
    ]
    summary_data = {}
    for k in keys:
        vals = [m[k] for m in fold_metrics_list]
        mean_v, std_v = np.mean(vals), np.std(vals)
        summary_data[k] = f"{mean_v:.3f} ± {std_v:.3f}"
        print(f"{k:<12}: {mean_v:.3f} ± {std_v:.3f}")

    # 6. 可选：保存为 5fold_results.pkl，供后续画多模型 ROC/PR 和计算 p 值使用
    if save_pkl_path is None:
        save_pkl_path = os.path.join(CONFIG['save_dir'], '5fold_results.pkl')
    os.makedirs(os.path.dirname(save_pkl_path), exist_ok=True)
    with open(save_pkl_path, 'wb') as f:
        pickle.dump({'fold_results': oof_fold_dict, 'fold_metrics': fold_metrics_list}, f)
    print(f"\n[已保存] 5 折预测结果已导出至: {save_pkl_path}")

    return fold_metrics_list, pd.DataFrame([summary_data])


# 直接运行推理
fold_metrics_youden, df_tabnet_summary = inference_5fold_with_youden(CONFIG)

TabNet 5-Fold Inference (Youden's Index Optimal Threshold)
[Dataset] 病人: 43, 关节样本: 86, 正类(III型): 31
Fold 1 | Youden Thresh: 0.5851 | AUC: 0.515 | AP: 0.429 | F1: 0.459 | Acc: 0.616 | Sen: 0.452 | Spe: 0.709
[Dataset] 病人: 43, 关节样本: 86, 正类(III型): 22
Fold 2 | Youden Thresh: 0.5060 | AUC: 0.750 | AP: 0.488 | F1: 0.652 | Acc: 0.814 | Sen: 0.682 | Spe: 0.859
[Dataset] 病人: 43, 关节样本: 86, 正类(III型): 30
Fold 3 | Youden Thresh: 0.6109 | AUC: 0.574 | AP: 0.487 | F1: 0.528 | Acc: 0.605 | Sen: 0.633 | Spe: 0.589
[Dataset] 病人: 43, 关节样本: 86, 正类(III型): 23
Fold 4 | Youden Thresh: 0.6484 | AUC: 0.608 | AP: 0.337 | F1: 0.468 | Acc: 0.709 | Sen: 0.478 | Spe: 0.794
[Dataset] 病人: 43, 关节样本: 86, 正类(III型): 29
Fold 5 | Youden Thresh: 0.4923 | AUC: 0.608 | AP: 0.406 | F1: 0.522 | Acc: 0.616 | Sen: 0.621 | Spe: 0.614

5-Fold Results Summary (Youden's Index Threshold)
threshold   : 0.569 ± 0.060
auc         : 0.611 ± 0.077
ap          : 0.430 ± 0.056
f1          : 0.526 ± 0.069
specificity : 0.713 ± 0.103
sensitivit